### 1. Import Libraries
Import pandas for data handling, and scikit-learn tools for splitting data, building the logistic regression model, and evaluating its performance.then load the dataset


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_excel(r"C:\Users\Shahe\Downloads\IBMHR-Employee-Attritionanalysis (2).xlsx", sheet_name="in")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,Promotion_Band,income_band,Tenure Band,Education.1,environment_satisfaction,job_satisfaction,JOB_INVOLVMENT,RELATIONSHIP_SATISFACTION,worklife_balance,performance_rating
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,00-Same Year,01-3001-6000,0-2,College,Medium,Very High,High,Low,Bad,Excellent
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,01-02,01-3001-6000,0-2,Below College,High,Medium,Medium,Very High,Better,Outstanding
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,00-Same Year,00-Under 3000,0-2,College,Very High,High,Medium,Medium,Better,Excellent
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,03-05,00-Under 3000,3-5,Masters,Very High,High,High,High,Better,Excellent
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,01-02,01-3001-6000,0-2,Below College,Low,Medium,High,Very High,Better,Excellent


### 2. Verify Data is Clean
Confirm the dataset has exactly 1470 rows before proceeding.

In [2]:
print(df.shape)

(1470, 48)


### 3. Check Column Names
List all column names in the dataset to confirm the exact spelling of key columns (such as the attrition flag) before using them in later code.

In [3]:
print(df.columns.tolist())

['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'attrition_flag', 'Age_Group', 'Distance_Band', 'Promotion_Band', 'income_band', 'Tenure Band', 'Education.1', 'environment_satisfaction', 'job_satisfaction', 'JOB_INVOLVMENT', 'RELATIONSHIP_SATISFACTION', 'worklife_balance', 'performance_rating']


### 4. Encode Categorical Variables
Convert text-based columns (OverTime, Department, MaritalStatus, BusinessTravel, Gender, JobRole, EducationField) into numeric 0/1 columns using one-hot encoding, since logistic regression requires numeric input rather than text.

In [4]:
df_encoded = pd.get_dummies(df, columns=['OverTime', 'Department', 'MaritalStatus', 'BusinessTravel', 'Gender', 'JobRole', 'EducationField'], drop_first=True)

###5. Define Input Features (X) and Target Variable (y)
Select the input factors believed to influence attrition, based on patterns identified in the earlier Excel analysis (income, satisfaction scores, distance from home, tenure, promotion history, overtime, and age). Set attrition_flag as the target variable, the outcome the model will learn to predict.

In [5]:
X = df_encoded[['MonthlyIncome', 'JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance', 
                 'DistanceFromHome', 'YearsAtCompany', 'YearsSinceLastPromotion', 
                 'OverTime_Yes', 'Age']]
y = df_encoded['attrition_flag']

In [6]:
print(X.isnull().sum())

MonthlyIncome              0
JobSatisfaction            0
EnvironmentSatisfaction    0
WorkLifeBalance            0
DistanceFromHome           0
YearsAtCompany             0
YearsSinceLastPromotion    0
OverTime_Yes               0
Age                        0
dtype: int64


### 5. Define Input Features (X) and Target Variable (y)
Select the input factors believed to influence attrition (income, satisfaction scores, distance from home, tenure, promotion history, overtime, and age). Set attrition_flag as the target variable the model will learn to predict.

In [7]:
X = df_encoded[['MonthlyIncome', 'JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance', 
                 'DistanceFromHome', 'YearsAtCompany', 'YearsSinceLastPromotion', 
                 'OverTime_Yes', 'Age']]
y = df_encoded['attrition_flag']

### 6. Split Data into Training and Testing Sets
Divide the dataset into an 80% training set (used to teach the model) and a 20% testing set (held back to evaluate the model on employees it has never seen). A fixed random seed ensures the same split is produced every time the code is run.

In [8]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)

(1176, 9)
(294, 9)


### 7. Initial Model (No Class Balancing)
Train a baseline logistic regression model without addressing class imbalance. While overall accuracy appears high, this model performs poorly at identifying employees who actually leave, since only about 16% of the dataset represents attrition cases.

In [9]:
model_unbalanced = LogisticRegression(max_iter=1000)
model_unbalanced.fit(X_train, y_train)
y_pred_unbalanced = model_unbalanced.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_unbalanced))
print(classification_report(y_test, y_pred_unbalanced))

Accuracy: 0.8605442176870748
              precision    recall  f1-score   support

           0       0.87      0.98      0.92       255
           1       0.38      0.08      0.13        39

    accuracy                           0.86       294
   macro avg       0.62      0.53      0.53       294
weighted avg       0.81      0.86      0.82       294



### 8. Balanced Model (Addressing Class Imbalance)
Retrain the model using class_weight='balanced' to give more weight to the minority class (employees who left). This trades some overall accuracy for a significant improvement in correctly identifying at-risk employees, better aligning with the goal of proactive retention.

In [10]:
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_balanced))
print(classification_report(y_test, y_pred_balanced))

Accuracy: 0.717687074829932
              precision    recall  f1-score   support

           0       0.90      0.76      0.82       255
           1       0.23      0.46      0.30        39

    accuracy                           0.72       294
   macro avg       0.56      0.61      0.56       294
weighted avg       0.81      0.72      0.75       294



### 9 Testing a Lower Decision Threshold (0.3)
Explore whether lowering the classification threshold from the default 0.5 to 0.3 improves the model's ability to catch at-risk employees. This trades precision (more false alarms) for higher recall (catching more actual leavers), a tradeoff worth testing given the cost of missing a genuine flight risk.

In [11]:
y_prob = model_balanced.predict_proba(X_test)[:, 1]
y_pred_threshold03 = (y_prob >= 0.3).astype(int)
print(classification_report(y_test, y_pred_threshold03))

              precision    recall  f1-score   support

           0       0.96      0.42      0.58       255
           1       0.19      0.87      0.31        39

    accuracy                           0.48       294
   macro avg       0.57      0.65      0.45       294
weighted avg       0.85      0.48      0.55       294



### 10. Generate Attrition Risk Scores
Use the balanced model to calculate a predicted probability of leaving for every employee in the dataset, creating an individual risk score rather than just a group-level average.

In [12]:
df_encoded['Attrition_Probability'] = model_balanced.predict_proba(X)[:, 1]

### Preview the Risk Scores
Check the first 10 employees to confirm the Attrition_Probability column populated correctly.

In [13]:
df_encoded[['EmployeeNumber', 'Attrition_Probability']].head(10)

,EmployeeNumber,Attrition_Probability
0,1,0.570037
1,2,0.280599
2,4,0.691555
3,5,0.674605
4,7,0.480000
5,8,0.243311
6,10,0.766154
7,11,0.409872
8,12,0.217182
9,13,0.522822


### 11. Identify Highest-Risk Employees
Sort employees by their predicted attrition probability, from highest to lowest, to surface the individuals HR should prioritize for retention efforts.

In [14]:
df_encoded[['EmployeeNumber', 'Attrition_Probability']].sort_values(by='Attrition_Probability', ascending=False).head(10)

,EmployeeNumber,Attrition_Probability
1058,1489,0.915096
798,1108,0.907181
889,1244,0.901939
514,702,0.896072
1313,1844,0.892987
398,529,0.889295
658,913,0.886551
26,33,0.872414
277,382,0.871913
892,1248,0.861358


### 12. Export Risk Scores for Excel Integration
Save the EmployeeNumber and Attrition_Probability columns to a CSV file, to be brought back into the main Excel workbook via XLOOKUP for building risk tier classifications.

In [15]:
df_encoded[['EmployeeNumber', 'Attrition_Probability']].to_csv('attrition_risk_scores.csv', index=False)

In [16]:
import os
print(os.getcwd())
print(os.listdir())

D:\
['$RECYCLE.BIN', '.ipynb_checkpoints', 'attrition_risk_scores.csv', 'kakkuuzzz', 'kakunte kutti', 'New folder', 'New folder (2)', 'shaheer', 'System Volume Information', 'Untitled.ipynb', 'Untitled1.ipynb']
